# Advanced Problems: Floats - Equality Testing (Deep Dive)

This notebook contains advanced, interview-level and real-world problems involving floating-point equality.

Topics covered:
- Accumulation error
- Relative vs absolute tolerance
- Robust comparisons
- Numerical stability
- Catastrophic cancellation
- IEEE-754 edge cases (NaN, inf)
- Financial precision issues
- Testing strategies

Python Version: 3.13

## Problem 1: Massive Accumulation Drift

Accumulate `0.1` ten million times.

Tasks:
1. Compare result to expected value.
2. Measure absolute and relative error.
3. Fix using `math.isclose`.
4. Explain scaling of error.

In [1]:
from math import isclose

total = 0.0
n = 1_000_000
for _ in range(n):
    total += 0.1

expected = n * 0.1

abs_error = abs(total - expected)
rel_error = abs_error / expected

print("== comparison:", total == expected)
print("Absolute error:", abs_error)
print("Relative error:", rel_error)
print("isclose:", isclose(total, expected, rel_tol=1e-9))

== comparison: False
Absolute error: 1.3328826753422618e-06
Relative error: 1.3328826753422619e-11
isclose: True


## Problem 2: Precision Loss in Summation Order

Sum numbers in different orders:
- small to large
- large to small

Explain the difference.

In [2]:
nums = [1e16, 1.0, -1e16]

sum1 = sum(nums)
sum2 = nums[1] + nums[0] + nums[2]

print("Order 1:", sum1)
print("Order 2:", sum2)

# Floating-point addition is NOT associative.

Order 1: 1.0
Order 2: 0.0


## Problem 3: Robust Comparison Utility (Production-Ready)

Build a function that:
- Handles NaN safely
- Handles infinities
- Uses combined tolerances
- Optionally logs mismatches

In [3]:
import math

def robust_equal(a, b, rel_tol=1e-9, abs_tol=1e-12, debug=False):
    if math.isnan(a) or math.isnan(b):
        return False
    if math.isinf(a) or math.isinf(b):
        return a == b

    result = abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)

    if debug and not result:
        print(f"Mismatch: {a} vs {b}")

    return result

print(robust_equal(0.3, 0.1 + 0.1 + 0.1, debug=True))

True


## Problem 4: Floating-Point Bucketing (Clustering)

Group numbers that are approximately equal into buckets.

Constraint:
- Use tolerance-based grouping
- Preserve insertion order

In [4]:
from math import isclose

values = [0.3, 0.30000000000000004, 0.29, 0.29000000000000004]

buckets = []

for v in values:
    for bucket in buckets:
        if isclose(v, bucket[0], rel_tol=1e-9):
            bucket.append(v)
            break
    else:
        buckets.append([v])

print(buckets)

[[0.3, 0.30000000000000004], [0.29, 0.29000000000000004]]


## Problem 5: Relative vs Absolute Failure Case

Show a scenario where:
- Relative tolerance fails
- Absolute tolerance succeeds

Explain why.

In [5]:
from math import isclose

x = 1e-15
y = 2e-15

print("rel_tol only:", isclose(x, y, rel_tol=1e-9))
print("with abs_tol:", isclose(x, y, rel_tol=1e-9, abs_tol=1e-14))

rel_tol only: False
with abs_tol: True


## Problem 6: Catastrophic Cancellation

Evaluate `(a + b) - a` for large `a` and small `b`.

Explain precision loss.

In [6]:
a = 1e16
b = 1.0

result = (a + b) - a

print("Expected:", b)
print("Actual:", result)

Expected: 1.0
Actual: 0.0


## Problem 7: Financial Precision Failure

Simulate adding cents using floats.

Fix using:
- rounding
- Decimal

Explain differences.

In [7]:
from decimal import Decimal

total = 0.0
for _ in range(1000):
    total += 0.01

print("Float:", total)
print("Rounded:", round(total, 2))

total_dec = Decimal('0.00')
for _ in range(1000):
    total_dec += Decimal('0.01')

print("Decimal:", total_dec)

Float: 9.999999999999831
Rounded: 10.0
Decimal: 10.00


## Problem 8: Detect Equality Drift Over Time

Simulate iterative computation where error accumulates.

Detect when values diverge beyond tolerance.

In [8]:
from math import isclose

x = 1.0
y = 1.0

for i in range(1_000_000):
    x += 0.000001
    y += 0.0000010000000001

print("Close?", isclose(x, y, rel_tol=1e-9))

Close? True


## Problem 9: IEEE-754 Edge Cases

Investigate behavior of:
- NaN
- inf

Compare using `==` and `isclose`.

In [9]:
from math import isclose

nan = float('nan')
inf = float('inf')

print("nan == nan:", nan == nan)
print("isclose(nan, nan):", isclose(nan, nan))

print("inf == inf:", inf == inf)
print("isclose(inf, inf):", isclose(inf, inf))

nan == nan: False
isclose(nan, nan): False
inf == inf: True
isclose(inf, inf): True


## Problem 10: Unit Testing Float Comparisons

Write assertions for float comparisons.

Use:
- math.isclose
- pytest-style checks

In [10]:
from math import isclose

def test_float():
    assert isclose(0.1 + 0.1 + 0.1, 0.3, rel_tol=1e-9)

test_float()
print("Test passed")

Test passed
